In [3]:
! pip install yt_dlp
! pip install python-ffmpeg


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [python-ffmpeg]


In [4]:
#!/usr/bin/env python3
"""Download a YouTube playlist into a clean, ordered folder structure."""

from __future__ import annotations

import argparse
import sys
from pathlib import Path


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Download a YouTube playlist in playlist order."
    )
    parser.add_argument("url", help="YouTube playlist URL")
    parser.add_argument(
        "-o",
        "--output",
        type=Path,
        default=Path("downloads"),
        help="Destination directory (default: ./downloads)",
    )
    parser.add_argument(
        "--audio-only",
        action="store_true",
        help="Save audio as MP3 instead of downloading video",
    )
    parser.add_argument(
        "--audio-quality",
        choices=["0", "2", "5", "7", "9"],
        default="2",
        help="MP3 VBR quality: 0 is best, 9 is smallest (default: 2)",
    )
    parser.add_argument(
        "--resolution",
        type=int,
        default=1080,
        help="Maximum video height, such as 720 or 1080 (default: 1080)",
    )
    parser.add_argument(
        "--subtitles",
        action="store_true",
        help="Download English subtitles when available",
    )
    parser.add_argument(
        "--cookies-from-browser",
        choices=["brave", "chrome", "chromium", "edge", "firefox", "opera", "safari", "vivaldi"],
        help="Use cookies from a browser for videos you can access there",
    )
    parser.add_argument(
        "--start",
        type=int,
        metavar="N",
        help="Start at playlist item N",
    )
    parser.add_argument(
        "--end",
        type=int,
        metavar="N",
        help="Stop at playlist item N",
    )
    return parser.parse_args()


def build_options(args: argparse.Namespace) -> dict:
    output_dir = args.output.expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    # The video ID prevents collisions when two videos have the same title.
    output_template = str(
        output_dir
        / "%(playlist_title|Single Videos)s"
        / "%(playlist_index)03d - %(title)s [%(id)s].%(ext)s"
    )

    options: dict = {
        "outtmpl": output_template,
        "download_archive": str(output_dir / ".downloaded.txt"),
        "ignoreerrors": True,
        "continuedl": True,
        "noplaylist": False,
        "windowsfilenames": True,
        "writedescription": True,
        "writeinfojson": True,
        "writethumbnail": True,
        "embedmetadata": True,
        "embedthumbnail": True,
        "retries": 10,
        "fragment_retries": 10,
        "sleep_interval": 1,
        "max_sleep_interval": 3,
        "quiet": False,
        "no_warnings": False,
    }

    if args.audio_only:
        options.update(
            {
                "format": "bestaudio/best",
                "postprocessors": [
                    {
                        "key": "FFmpegExtractAudio",
                        "preferredcodec": "mp3",
                        "preferredquality": args.audio_quality,
                    },
                    {"key": "FFmpegMetadata"},
                    {"key": "EmbedThumbnail"},
                ],
            }
        )
    else:
        options.update(
            {
                "format": (
                    f"bestvideo[height<={args.resolution}]+bestaudio/"
                    f"best[height<={args.resolution}]/best"
                ),
                "merge_output_format": "mp4",
            }
        )

    if args.subtitles:
        options.update(
            {
                "writesubtitles": True,
                "writeautomaticsub": True,
                "subtitleslangs": ["en", "en.*"],
                "subtitlesformat": "srt/best",
            }
        )

    if args.cookies_from_browser:
        options["cookiesfrombrowser"] = (args.cookies_from_browser,)
    if args.start is not None:
        options["playliststart"] = args.start
    if args.end is not None:
        options["playlistend"] = args.end

    return options


def validate_args(args: argparse.Namespace) -> None:
    if args.resolution <= 0:
        raise SystemExit("--resolution must be greater than 0")
    if args.start is not None and args.start <= 0:
        raise SystemExit("--start must be greater than 0")
    if args.end is not None and args.end <= 0:
        raise SystemExit("--end must be greater than 0")
    if args.start is not None and args.end is not None and args.start > args.end:
        raise SystemExit("--start cannot be greater than --end")


def main() -> int:
    args = parse_args()
    validate_args(args)
    options = build_options(args)

    try:
        import yt_dlp
    except ImportError:
        print(
            "Missing dependency: yt-dlp\n"
            "Install it with: python3 -m pip install -r requirements.txt",
            file=sys.stderr,
        )
        return 1

    print(f"Destination: {args.output.expanduser().resolve()}")
    print("Starting playlist download. Press Ctrl+C to stop safely.\n")

    try:
        with yt_dlp.YoutubeDL(options) as downloader:
            error_code = downloader.download([args.url])
    except KeyboardInterrupt:
        print("\nStopped. Run the same command later to resume.", file=sys.stderr)
        return 130
    except yt_dlp.utils.DownloadError as error:
        print(f"\nDownload failed: {error}", file=sys.stderr)
        return 1

    if error_code:
        print("\nFinished with one or more skipped/failed items.", file=sys.stderr)
        return 1

    print("\nPlaylist download complete.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


usage: ipykernel_launcher.py [-h] [-o OUTPUT] [--audio-only]
                             [--audio-quality {0,2,5,7,9}]
                             [--resolution RESOLUTION] [--subtitles]
                             [--cookies-from-browser {brave,chrome,chromium,edge,firefox,opera,safari,vivaldi}]
                             [--start N] [--end N]
                             url
ipykernel_launcher.py: error: the following arguments are required: url


SystemExit: 2

In [ ]:
dpython yp.py "https://www.youtube.com/playlist?list=PLlitOiy7LlnsOjeDJpWSpQXIshfUPgWU1&si=lk3-vgwXgN04mCmo"

SyntaxError: invalid syntax (3478617924.py, line 1)